# Rule-based baselines

This notebook runs five rule-based string-matching methods as the comparison
for the neural pipeline: RapidFuzz ratio, Levenshtein (normalised
similarity), Jaro-Winkler, TF-IDF cosine on character 3-grams, and Double
Metaphone codes compared by the RapidFuzz ratio. Part 1 lets each method
search all 15,901,232 G-NAF records. Part 2 runs the same five methods again
inside a street-number block. Every method sees the same test set, the same
rendered record strings and the same correctness rule as the pipeline.

- Reads: the test set, gnaf.duckdb, the retriever's model, index and per-query results, the re-ranker's model and its saved predictions.
- Writes: `output_baseline/` (Part 1) and `output_baseline_blocked/` (Part 2).
- Run after: 4_train_reranker.ipynb

The full-search runs in Cell 9 take about 25 minutes. Setting B_SMOKE=1
searches the first 300,000 records plus every true record instead, as a quick
test of the code path.

In [ ]:
# Cell 1: configuration. Every path can be overridden from the environment.
# S1_OUT and S2_MODEL point at the retriever's and re-ranker's outputs,
# B_S2PRED at the re-ranker's saved predictions, B_MISS at the test set,
# B_DB at the database and B_OUT at the output folder. B_SMOKE=1 searches a
# 300,000-row pool and writes to scratch/baseline_smoke/ (a code-path test).
# The output folder may not sit inside any input folder.

import os, sys, json, time, re, hashlib
from pathlib import Path
import numpy as np, pandas as pd

def _env(name, default): return os.environ.get(name, default)

S1_OUT   = Path(_env("S1_OUT", "output_stage1"))
S1_MODEL = S1_OUT / "model"                                   # the retriever
EMB_NPY  = S1_OUT / "emb_store.npy"                           # 15,901,232 x 384 fp32
PID_NPY  = S1_OUT / "emb_pids.npy"                            # row i of EMB_NPY is pid i
S1_PERQ  = S1_OUT / "stage1_recall_per_query.tsv"             # the 789 matchable queries with truth_pid
S2_MODEL = Path(_env("S2_MODEL", "output_stage2/model"))      # the re-ranker
S2_PRED  = Path(_env("B_S2PRED", "results_labels_789_model_refit/stage2/stage2_predictions.tsv"))
S2_MANIF = S2_PRED.parent / "stage2_run_manifest.json"        # carries the pipeline's tau
DB_PATH  = Path(_env("B_DB", "gnaf.duckdb"))
MISS     = Path(_env("B_MISS", "output/misspellings.tsv"))
EXPECTED_TEST_SHA1 = "d48fdfe227fd6ef7828384f020e3042d18873101"  # sha1 of the test set

SMOKE = _env("B_SMOKE", "0") == "1"
OUT   = Path(_env("B_OUT", "scratch/baseline_smoke" if SMOKE else "output_baseline"))
PROTECTED = ["output", "output_stage1", "output_stage2", "output_synthesis", "harvest",
             "cache", "archive", "results_labels_746_model_85k",
             "results_labels_789_model_85k", "results_labels_789_model_refit"]
_o = OUT.resolve()
for _p in PROTECTED:
    _r = Path(_p).resolve()
    assert _o != _r and _r not in _o.parents, f"OUT={OUT} is inside protected {_p}; refusing"
OUT.mkdir(parents=True, exist_ok=True)

SEED        = 4640           # the project seed, used for the fold split and the timed queries
K_KEEP      = 50             # candidates kept per query per method (recall@50 needs 50)
CHUNK_ROWS  = 250_000        # pool rows scored per block (950 x 250k x 4 B = 0.95 GB)
N_THREADS   = 8              # threads for the one-query-alone timing
N_SINGLE    = 5              # queries timed one at a time
RETRIEVE_K  = 500            # retriever depth, as in 4_train_reranker
K_REPORT    = 10             # re-ranker depth of the reported result
EMBED_BATCH = 64             # query encode batch, as in 4_train_reranker
PREDICT_BATCH = 128          # re-ranker batch, as in 4_train_reranker
SMOKE_POOL  = 300_000
TFIDF_FEATURES = 2 ** 17     # hashing width (number of 3-gram slots)
TFIDF_NGRAM    = (3, 3)
FOLLOW = "25 Hoskins Avenue, Kemblawarra NSW 2505"

for label, p in [("Stage 1 model", S1_MODEL), ("pid array", PID_NPY), ("index", EMB_NPY),
                 ("Stage 1 per-query", S1_PERQ), ("Stage 2 model", S2_MODEL),
                 ("neural predictions", S2_PRED), ("database", DB_PATH), ("test set", MISS)]:
    assert p.exists(), f"{label} not found at {p}"
print(f"OUT      : {OUT}   smoke={SMOKE}")
print(f"test set : {MISS}")
print(f"neural   : {S2_PRED}")

OUT      : output_baseline   smoke=False
test set : output/misspellings.tsv
neural   : results_labels_789_model_refit/stage2/stage2_predictions.tsv


In [ ]:
# Cell 2: open the database read-only, restate the record renderer RENDER
# (its sha1 is asserted) and define norm(), the query normaliser the
# retriever and re-ranker use: upper case, commas to spaces, single spaces.
# Every method sees exactly these strings.

import duckdb
con = duckdb.connect(str(DB_PATH), read_only=True)

RENDER = r"""regexp_replace(trim(concat_ws(' ',
    CASE WHEN g.number_last IS NOT NULL AND g.number_last <> ''
              AND regexp_matches(k.k, '^[0-9]')
         THEN k.k || '-' || g.number_last ELSE k.k END,
    k.street_n, k.type_n, k.suffix_n, k.locality_n, k.state_n, k.postcode)),
    '\s+', ' ', 'g')"""
RJOIN = "LEFT JOIN gnaf_range g ON g.pid = k.pid AND g.number_first = k.k"
RENDER_SHA1 = hashlib.sha1(RENDER.encode()).hexdigest()
assert RENDER_SHA1 == "cc573253b3ead7dddebc8393697909d49f27eaa1", "RENDER differs from the protected one"

def norm(s):
    """The query normaliser the retriever uses: upper, commas to spaces, collapse."""
    return re.sub(r"\s+", " ", str(s).upper().replace(",", " ")).strip()

def resolve(pids):
    """pid -> (rendered string, principal pid) exactly as 4_train_reranker does:
    prefer the non-LOT key, min() over renders; principal from PRINCIPAL_PID."""
    con.register("need_df", pd.DataFrame({"pid": sorted(set(pids))}))
    return con.sql(f"""
        SELECT n.pid,
               coalesce(min(CASE WHEN k.k NOT LIKE 'LOT %' THEN {RENDER} END),
                        min({RENDER}))                                           AS s,
               min(coalesce(nullif(CAST(k.PRINCIPAL_PID AS VARCHAR), ''), n.pid)) AS principal
        FROM need_df n JOIN gnaf_key k ON k.pid = n.pid {RJOIN}
        GROUP BY n.pid""").df()

print(f"database open read-only; RENDER sha1 {RENDER_SHA1[:8]} ok")
print(f"norm({FOLLOW!r}) = {norm(FOLLOW)!r}")

database open read-only; RENDER sha1 cc573253 ok
norm('25 Hoskins Avenue, Kemblawarra NSW 2505') = '25 HOSKINS AVENUE KEMBLAWARRA NSW 2505'


In [ ]:
# Cell 3: the 950 test queries in one frame, built the way 4_train_reranker
# builds them. Matchable rows (address, truth_pid) come from the retriever's
# per-query file. Unmatchable rows come from the test set's reason column:
# no_record is "Cannot be matched", no_number is "Street number does not
# exist". Rows marked excluded in the manual label review are kept but not
# scored (there are none).

TEST_SHA1 = hashlib.sha1(MISS.read_bytes()).hexdigest()
if MISS == Path("output/misspellings.tsv"):
    assert TEST_SHA1 == EXPECTED_TEST_SHA1, f"test set sha1 {TEST_SHA1} is not the expected test set"
full = pd.read_csv(MISS, sep="\t", dtype=str).fillna("")
assert full.address.is_unique, "addresses in the test set are not unique; merges below would be ambiguous"
_exc = (full.adjudication.str.contains("excluded", regex=False) if "adjudication" in full.columns
        else pd.Series(False, index=full.index))
no_rec = full.reason.str.contains("Cannot be matched") & ~_exc
no_num = full.reason.str.contains("Street number does not exist") & ~_exc

perq = pd.read_csv(S1_PERQ, sep="\t", dtype={"truth_pid": str})
match = perq[["address", "truth_pid"]].copy()
match["population"], match["kind"] = "matchable", "matchable"
unmatch = full[no_rec | no_num][["address"]].copy()
unmatch["truth_pid"] = ""
unmatch["population"] = "unmatchable"
unmatch["kind"] = np.where(full.loc[unmatch.index, "reason"].str.contains("Cannot be matched"),
                           "no_record", "no_number")
excl = full[_exc][["address"]].copy()
excl["truth_pid"], excl["population"], excl["kind"] = "", "excluded", "excluded"

allq = pd.concat([match, unmatch, excl], ignore_index=True)
allq["q"] = allq.address.map(norm)
allq["uid"] = allq.index
assert len(allq) == len(full), f"{len(allq)} queries assembled, file has {len(full)}"
assert set(allq.address) == set(full.address), "the three populations do not cover the file exactly"
assert allq.address.is_unique
m_uids = allq.loc[allq.population == "matchable", "uid"].to_numpy()
u_uids = allq.loc[allq.population == "unmatchable", "uid"].to_numpy()
FOLLOW_UID = int(allq.loc[allq.address == FOLLOW, "uid"].iloc[0])

print(f"test set sha1 {TEST_SHA1[:12]}")
for k, g in allq.groupby("kind"):
    print(f"  {k:12s} {len(g):5d}")
print(f"  {'TOTAL':12s} {len(allq):5d}")
print(f"followed address is uid {FOLLOW_UID}: {FOLLOW!r} -> q {allq.q[FOLLOW_UID]!r}")

test set sha1 d48fdfe227fd
  matchable      789
  no_number      129
  no_record       32
  TOTAL          950
followed address is uid 773: '25 Hoskins Avenue, Kemblawarra NSW 2505' -> q '25 HOSKINS AVENUE KEMBLAWARRA NSW 2505'


In [ ]:
# Cell 4: the search pool, one rendered string per G-NAF record. Every record
# of the retriever's index (table s1_index_key, one key per pid) is rendered
# with RENDER in pid order, and the pids are checked against emb_pids.npy row
# for row, so the methods search exactly the strings the retriever embedded.
# The pool is cached to OUT/pool_strings.parquet with a sidecar file (row
# count, sha1 of the pid file, sha1 of RENDER) and rebuilt if any differs.
# In smoke mode the pool is the first SMOKE_POOL rows plus every true record.

import pyarrow as pa, pyarrow.parquet as pq
PREP = {}
P_INDEX = np.load(PID_NPY)
N_INDEX = len(P_INDEX)
assert bool((P_INDEX[:-1] <= P_INDEX[1:]).all()), "index pids are not sorted; searchsorted lookups would be wrong"
PID_SHA1 = hashlib.sha1(PID_NPY.read_bytes()).hexdigest()
POOL_PARQ, POOL_SIDE = OUT / "pool_strings.parquet", OUT / "pool_strings.json"

_POOL_SQL = f"""
    SELECT i.pid, {RENDER} AS s
    FROM s1_index_key i
    JOIN gnaf_key k ON k.pid = i.pid AND k.k = i.k
    {RJOIN}
    {{where}}
    ORDER BY i.pid {{limit}}"""

if SMOKE:
    _truth = allq.loc[allq.population == "matchable", "truth_pid"].tolist()
    con.register("truth_df", pd.DataFrame({"pid": sorted(set(_truth))}))
    t0 = time.perf_counter()
    _a = con.execute(_POOL_SQL.format(where="", limit=f"LIMIT {SMOKE_POOL}")).to_arrow_table()
    _b = con.execute(_POOL_SQL.format(where="WHERE i.pid IN (SELECT pid FROM truth_df)", limit="")).to_arrow_table()
    _df = pd.concat([_a.to_pandas(), _b.to_pandas()]).drop_duplicates("pid").sort_values("pid")
    PREP["render_s"] = time.perf_counter() - t0
    POOL = _df.s.tolist(); POOL_PIDS = _df.pid.to_numpy().astype("S32")
    assert np.isin(POOL_PIDS, P_INDEX).all(), "smoke pool has a pid the index lacks"
    print(f"[smoke] pool of {len(POOL):,} rows ({SMOKE_POOL:,} + truth rows) in {PREP['render_s']:.1f}s")
else:
    _fresh, _why = True, "no cache"
    if POOL_PARQ.exists() and POOL_SIDE.exists():
        _sc = json.loads(POOL_SIDE.read_text())
        _checks = {"rows": _sc.get("n_rows") == N_INDEX, "pid file": _sc.get("pid_sha1") == PID_SHA1,
                   "RENDER": _sc.get("render_sha1") == RENDER_SHA1}
        _fresh = not all(_checks.values())
        _why = ", ".join(k for k, ok in _checks.items() if not ok) or "all checks pass"
    if _fresh:
        print(f"rendering the pool ({_why}) ...")
        t0 = time.perf_counter()
        _tab = con.execute(_POOL_SQL.format(where="", limit="")).to_arrow_table()
        PREP["render_s"] = time.perf_counter() - t0
        pq.write_table(_tab, POOL_PARQ, compression="zstd")
        POOL_SIDE.write_text(json.dumps(dict(n_rows=_tab.num_rows, pid_sha1=PID_SHA1,
                                             render_sha1=RENDER_SHA1, render_s=round(PREP["render_s"], 1),
                                             written=time.strftime("%Y-%m-%d %H:%M:%S"))))
        print(f"  rendered {_tab.num_rows:,} rows in {PREP['render_s']/60:.1f} min -> {POOL_PARQ} "
              f"({POOL_PARQ.stat().st_size/2**20:.0f} MiB)")
        del _tab
    else:
        PREP["render_s"] = json.loads(POOL_SIDE.read_text())["render_s"]
        print(f"pool cache is current ({_why}); render took {PREP['render_s']/60:.1f} min when written")
    t0 = time.perf_counter()
    _tab = pq.read_table(POOL_PARQ)
    POOL_PIDS = _tab.column("pid").to_numpy().astype("S32")
    POOL = _tab.column("s").to_pylist()
    del _tab
    PREP["load_s"] = time.perf_counter() - t0
    assert len(POOL) == N_INDEX and np.array_equal(POOL_PIDS, P_INDEX), \
        "pool pids differ from the index pids; the baselines would not search what the bi-encoder indexed"
    print(f"pool loaded in {PREP['load_s']:.0f}s: {len(POOL):,} strings; pids equal the index row for row: True")

N_POOL = len(POOL)
def row_of_pid(pids):
    """rows of these pids in POOL (bytes S32 in), asserting each is present."""
    p = np.asarray(pids, dtype="S32"); r = np.searchsorted(POOL_PIDS, p)
    r = np.minimum(r, N_POOL - 1)
    assert (POOL_PIDS[r] == p).all(), "a pid is not in the pool"
    return r
_r = row_of_pid([P_INDEX[0]]); print(f"first pool row: {POOL_PIDS[0].decode()}  {POOL[0]!r}")

pool cache is current (all checks pass); render took 0.1 min when written


pool loaded in 11s: 15,901,232 strings; pids equal the index row for row: True
first pool row: GAACT714845933  '6 PACKHAM PLACE CHARNWOOD ACT 2615'


In [ ]:
# Cell 5: the true record's pool string for each matchable query (a method's
# top-1 is right when it equals this string), checked against the
# re-ranker's rendering of the same pid. Then load the re-ranker's saved
# top-1 per query and check its is_truth flag against the same string rule.

_tp = allq.loc[m_uids, "truth_pid"].to_numpy().astype("S32")
_rows = row_of_pid(_tp)
TRUTH_STR = {int(u): POOL[r] for u, r in zip(m_uids, _rows)}
_res = resolve([p.decode() for p in _tp])
_s2 = dict(zip(_res.pid, _res.s)); TRUTH_PRINCIPAL = dict(zip(_res.pid, _res.principal))
_diff = [u for u in m_uids if TRUTH_STR[u] != _s2[allq.truth_pid[u]]]
print(f"truth strings: {len(TRUTH_STR)}; differ from the Stage 2 rendering: {len(_diff)}")
for u in _diff[:5]:
    print(f"   uid {u}: pool {TRUTH_STR[u]!r} vs stage2 {_s2[allq.truth_pid[u]]!r}")
print(f"followed address truth: {TRUTH_STR[FOLLOW_UID]!r}")

neural_top1 = pd.read_csv(S2_PRED, sep="\t", dtype={"pid": str})
neural_top1 = neural_top1.merge(allq[["uid", "address", "population", "kind"]].rename(
    columns={"uid": "uid_here", "population": "pop_here", "kind": "kind_here"}), on="address", how="inner")
assert len(neural_top1) == len(allq) - int((allq.population == "excluded").sum()), \
    f"neural file has {len(neural_top1)} rows matched by address; expected {len(allq)}"
assert (neural_top1.population == neural_top1.pop_here).all(), "populations disagree with the neural file"
neural_top1["is_truth_here"] = [c == TRUTH_STR.get(int(u), "\x00")
                                for c, u in zip(neural_top1.cand, neural_top1.uid_here)]
_dis = int((neural_top1.is_truth_here != neural_top1.is_truth.astype(bool)).sum())
print(f"neural top-1 rows: {len(neural_top1)}; is_truth recomputed here disagrees with the file on {_dis} rows")
assert _dis == 0, "the string rule here does not reproduce the archived is_truth; stop and look"
TAU_PIPELINE = float(json.loads(S2_MANIF.read_text())["eval"]["tau"]) if S2_MANIF.exists() else float("nan")
print(f"pipeline tau from {S2_MANIF.name}: {TAU_PIPELINE:.2f}")

truth strings: 789; differ from the Stage 2 rendering: 0
followed address truth: '25 HOSKINS AVENUE WARRAWONG NSW 2502'
neural top-1 rows: 950; is_truth recomputed here disagrees with the file on 0 rows
pipeline tau from stage2_run_manifest.json: 0.83


In [ ]:
# Cell 6: full search for the RapidFuzz scorers. full_search() scores every
# query against the pool in blocks of CHUNK_ROWS on all cores and keeps a
# running top-K per query. single_search() times one query alone over the
# pool pre-split into N_THREADS slices, one thread per slice.
# Within the kept K, equal scores are ordered by pool row. At the K boundary
# ties are broken arbitrarily, which can affect recall@50 only.

from rapidfuzz import fuzz, process
from rapidfuzz.distance import Levenshtein, JaroWinkler
from concurrent.futures import ThreadPoolExecutor

def _merge(run_s, run_i, cand_s, cand_i, k):
    all_s = np.concatenate([run_s, cand_s], axis=1); all_i = np.concatenate([run_i, cand_i], axis=1)
    sel = np.argpartition(-all_s, k - 1, axis=1)[:, :k]
    return np.take_along_axis(all_s, sel, axis=1), np.take_along_axis(all_i, sel, axis=1)

def _order(run_s, run_i):
    o = np.argsort(run_i, axis=1, kind="stable")
    run_s, run_i = np.take_along_axis(run_s, o, axis=1), np.take_along_axis(run_i, o, axis=1)
    o = np.argsort(-run_s, axis=1, kind="stable")
    return np.take_along_axis(run_s, o, axis=1), np.take_along_axis(run_i, o, axis=1)

def full_search(scorer, queries, pool, k=K_KEEP, chunk_rows=CHUNK_ROWS, scale=1.0):
    """-> (rows (nq,k) int64, scores (nq,k) float32 in 0..1, seconds)."""
    nq, n = len(queries), len(pool)
    run_s = np.full((nq, k), -np.inf, dtype=np.float32); run_i = np.zeros((nq, k), dtype=np.int64)
    t0 = time.perf_counter()
    for s in range(0, n, chunk_rows):
        sims = process.cdist(queries, pool[s:s + chunk_rows], scorer=scorer, workers=-1, dtype=np.float32)
        kk = min(k, sims.shape[1])
        part = np.argpartition(-sims, kk - 1, axis=1)[:, :kk]
        run_s, run_i = _merge(run_s, run_i, np.take_along_axis(sims, part, axis=1),
                              part.astype(np.int64) + s, k)
    run_s, run_i = _order(run_s, run_i)
    return run_i, (run_s * scale).astype(np.float32), time.perf_counter() - t0

def split_pool(pool, n=N_THREADS):
    m = len(pool); return [pool[i * m // n:(i + 1) * m // n] for i in range(n)]

def single_search(scorer, q, parts, scale=1.0):
    """one query alone over the pre-split pool -> (best row, score, seconds)."""
    t0 = time.perf_counter()
    with ThreadPoolExecutor(len(parts)) as ex:
        out = list(ex.map(lambda p: process.cdist([q], p, scorer=scorer, workers=1, dtype=np.float32)[0], parts))
    sims = np.concatenate(out); j = int(np.argmax(sims))
    return j, float(sims[j] * scale), time.perf_counter() - t0

print("full_search / single_search ready")

full_search / single_search ready


In [ ]:
# Cell 7: TF-IDF cosine on character 3-grams. Each string becomes a bag of
# 3-grams (char_wb, so "HOSKINS" gives " HO", "HOS", "OSK", ... "NS "),
# weighted by idf = log((1+N)/(1+df)) + 1 and L2-normalised, so a dot product
# is the cosine. The pool is never held as one matrix: pass 1 counts
# document frequencies, pass 2 transforms each block, scores it against all
# queries and keeps a running top-K. Build time and search time are timed
# separately. The one-query-alone time is that query's per-block products,
# summed, which is what a pre-built matrix would cost.

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.preprocessing import normalize
HV = HashingVectorizer(analyzer="char_wb", ngram_range=TFIDF_NGRAM, n_features=TFIDF_FEATURES,
                       alternate_sign=False, norm=None, dtype=np.float32)

def tfidf_fit_idf(pool, chunk_rows=CHUNK_ROWS):
    t0 = time.perf_counter(); df = np.zeros(TFIDF_FEATURES, dtype=np.int64)
    for s in range(0, len(pool), chunk_rows):
        df += np.bincount(HV.transform(pool[s:s + chunk_rows]).indices, minlength=TFIDF_FEATURES)
    idf = (np.log((1 + len(pool)) / (1 + df)) + 1).astype(np.float32)
    return idf, time.perf_counter() - t0

def _vec(strings, idf):
    X = HV.transform(strings); X.data *= idf[X.indices]
    return normalize(X, norm="l2", copy=False)

def tfidf_search(queries, pool, idf, single_qs, k=K_KEEP, chunk_rows=CHUNK_ROWS):
    nq, n = len(queries), len(pool)
    Qd = np.ascontiguousarray(_vec(queries, idf).toarray().T)          # (features, nq)
    qv = _vec(single_qs, idf).toarray()                                 # (n_single, features)
    run_s = np.full((nq, k), -np.inf, dtype=np.float32); run_i = np.zeros((nq, k), dtype=np.int64)
    build_s = search_s = 0.0
    single_s = np.zeros(len(single_qs)); single_best = [(-1, -np.inf)] * len(single_qs)
    for s in range(0, n, chunk_rows):
        t = time.perf_counter(); X = _vec(pool[s:s + chunk_rows], idf); build_s += time.perf_counter() - t
        t = time.perf_counter()
        sims = np.ascontiguousarray((X @ Qd).T)                          # (nq, chunk)
        kk = min(k, sims.shape[1])
        part = np.argpartition(-sims, kk - 1, axis=1)[:, :kk]
        run_s, run_i = _merge(run_s, run_i, np.take_along_axis(sims, part, axis=1),
                              part.astype(np.int64) + s, k)
        search_s += time.perf_counter() - t
        for j in range(len(single_qs)):
            t = time.perf_counter(); sc = X @ qv[j]; b = int(np.argmax(sc))
            if sc[b] > single_best[j][1]: single_best[j] = (b + s, float(sc[b]))
            single_s[j] += time.perf_counter() - t
    run_s, run_i = _order(run_s, run_i)
    return run_i, run_s, dict(build_s=build_s, search_s=search_s, single_s=single_s.tolist(),
                              single_best=single_best)

print("tfidf_fit_idf / tfidf_search ready")

tfidf_fit_idf / tfidf_search ready


In [ ]:
# Cell 8: Double Metaphone coding. Each word becomes a short code for how it
# sounds (HOSKINS -> HSKNS, KEMBLAWARRA -> KMPLR, WARRAWONG -> ARNK). A word
# holding a digit (25, 36-38, 1A, 2505) is kept as written, so numbers still
# count, and a word with no code keeps its letters. The coded strings are
# compared with the RapidFuzz ratio. The pool's distinct words are coded once.

from metaphone import doublemetaphone

def dm_code(tok):
    if any(ch.isdigit() for ch in tok):
        return tok
    p = doublemetaphone(tok)[0]
    return p if p else tok

def dm_encode_pool(pool):
    """-> (coded strings, seconds, number of distinct words)"""
    t0 = time.perf_counter()
    vocab = set()
    for s in pool:
        vocab.update(s.split())
    CODE = {w: dm_code(w) for w in vocab}
    coded = [" ".join([CODE[w] for w in s.split()]) for s in pool]
    return coded, time.perf_counter() - t0, len(vocab)

def dm_encode(strings):
    return [" ".join(dm_code(w) for w in s.split()) for s in strings]

print(f"{norm(FOLLOW)!r} -> {dm_encode([norm(FOLLOW)])[0]!r}")
print(f"{TRUTH_STR[FOLLOW_UID]!r} -> {dm_encode([TRUTH_STR[FOLLOW_UID]])[0]!r}")

'25 HOSKINS AVENUE KEMBLAWARRA NSW 2505' -> '25 HSKNS AFN KMPLR NS 2505'
'25 HOSKINS AVENUE WARRAWONG NSW 2502' -> '25 HSKNS AFN ARNK NS 2502'


In [ ]:
# Cell 9: run the five methods. For each, search all queries as one batch
# (top-K kept), then time N_SINGLE queries one at a time (the followed
# address plus four matchable queries drawn with the seed). Writes
# OUT/perquery_<method>.tsv (uid, rank, pid, cand, score) and OUT/timing.tsv.
# About 25 minutes in total on the full pool.

METHOD_LABEL = {"rapidfuzz_ratio": "RapidFuzz ratio", "levenshtein": "Levenshtein",
                "jaro_winkler": "Jaro-Winkler", "tfidf_char3": "TF-IDF cosine (character 3-grams)",
                "double_metaphone": "Double Metaphone + ratio",
                "neural": "bi-encoder + cross-encoder (K=10)"}
QUERIES = allq.q.tolist()
_rng = np.random.default_rng(SEED)
_others = [u for u in m_uids if u != FOLLOW_UID]
SINGLE_UIDS = [FOLLOW_UID] + sorted(_rng.choice(_others, size=N_SINGLE - 1, replace=False).tolist())
SINGLE_Q = [QUERIES[u] for u in SINGLE_UIDS]
print("one-query-alone timing on uids", SINGLE_UIDS)

RESULTS, TIMING = {}, []
def _pid_str(rows): return np.char.decode(POOL_PIDS[rows], "ascii")

def write_perquery(name, rows, scores, pool=None):
    pool = POOL if pool is None else pool
    nq, k = rows.shape
    df = pd.DataFrame({"uid": np.repeat(np.arange(nq), k), "rank": np.tile(np.arange(1, k + 1), nq),
                       "pid": _pid_str(rows.ravel()), "cand": [POOL[i] for i in rows.ravel()],
                       "score": scores.ravel()})
    df.to_csv(OUT / f"perquery_{name}.tsv", sep="\t", index=False)
    return df

def record(name, build_s, batch_s, singles, note=""):
    TIMING.append(dict(method=name, label=METHOD_LABEL[name], build_s=round(build_s, 3),
                       batch_s=round(batch_s, 3), per_query_ms=round(batch_s / len(QUERIES) * 1000, 2),
                       single_s_mean=round(float(np.mean(singles)), 4),
                       single_s_runs=json.dumps([round(x, 4) for x in singles]), note=note))
    print(f"  {METHOD_LABEL[name]:36s} build {build_s:7.1f}s | batch {batch_s:7.1f}s "
          f"({batch_s/len(QUERIES)*1000:6.1f} ms/query) | one alone {np.mean(singles):.3f}s")

PARTS = split_pool(POOL)
for name, scorer, scale in [("rapidfuzz_ratio", fuzz.ratio, 0.01),
                            ("levenshtein", Levenshtein.normalized_similarity, 1.0),
                            ("jaro_winkler", JaroWinkler.similarity, 1.0)]:
    rows, scores, secs = full_search(scorer, QUERIES, POOL, scale=scale)
    RESULTS[name] = (rows, scores); write_perquery(name, rows, scores)
    singles = []
    for u, q in zip(SINGLE_UIDS, SINGLE_Q):
        j, sc, dt = single_search(scorer, q, PARTS, scale=scale); singles.append(dt)
        assert j == rows[u, 0] or abs(sc - scores[u, 0]) < 1e-6, f"{name}: one-query search disagrees with the batch on uid {u}"
    record(name, 0.0, secs, singles, "no preparation needed")
del PARTS

idf, fit_s = tfidf_fit_idf(POOL)
rows, scores, tm = tfidf_search(QUERIES, POOL, idf, SINGLE_Q)
RESULTS["tfidf_char3"] = (rows, scores); write_perquery("tfidf_char3", rows, scores)
for (b, sc), u in zip(tm["single_best"], SINGLE_UIDS):
    assert b == rows[u, 0] or abs(sc - scores[u, 0]) < 1e-5, f"tfidf: one-query search disagrees on uid {u}"
record("tfidf_char3", fit_s + tm["build_s"], tm["search_s"], tm["single_s"],
       "build = idf pass + block transforms; search = products + top-K; one alone = per-block product only")

POOL_DM, dm_s, n_vocab = dm_encode_pool(POOL)
Q_DM = dm_encode(QUERIES); SINGLE_DM = dm_encode(SINGLE_Q)
rows, scores, secs = full_search(fuzz.ratio, Q_DM, POOL_DM, scale=0.01)
RESULTS["double_metaphone"] = (rows, scores); write_perquery("double_metaphone", rows, scores)
PARTS = split_pool(POOL_DM); singles = []
for u, q in zip(SINGLE_UIDS, SINGLE_DM):
    j, sc, dt = single_search(fuzz.ratio, q, PARTS, scale=0.01); singles.append(dt)
    assert j == rows[u, 0] or abs(sc - scores[u, 0]) < 1e-6, f"double_metaphone: one-query search disagrees on uid {u}"
record("double_metaphone", dm_s, secs, singles, f"build = encoding the pool ({n_vocab:,} distinct words)")
del PARTS, POOL_DM

pd.DataFrame(TIMING).to_csv(OUT / "timing.tsv", sep="\t", index=False)
print(f"\nwrote {len(RESULTS)} per-query files and timing.tsv to {OUT}")

one-query-alone timing on uids [773, 114, 125, 456, 723]


  RapidFuzz ratio                      build     0.0s | batch   189.2s ( 199.1 ms/query) | one alone 0.234s


  Levenshtein                          build     0.0s | batch   375.1s ( 394.9 ms/query) | one alone 0.410s


  Jaro-Winkler                         build     0.0s | batch   302.1s ( 318.0 ms/query) | one alone 0.334s


  TF-IDF cosine (character 3-grams)    build   210.4s | batch   176.2s ( 185.4 ms/query) | one alone 0.248s


  Double Metaphone + ratio             build    10.9s | batch   170.6s ( 179.6 ms/query) | one alone 0.207s

wrote 5 per-query files and timing.tsv to output_baseline


In [ ]:
# Cell 10: the neural pipeline timed on the same machine: retriever search
# over the full index (encode and scan timed separately), pid-to-string
# lookup and re-ranker scoring of the top 10, as a batch and for the
# N_SINGLE queries alone. Accuracy for the pipeline comes from its saved
# predictions (Cell 5). This cell only checks that it reproduces their
# top-1 picks. The index scan runs in numpy on the CPU over the 24 GB
# memory-mapped index, so one query alone costs a full pass (about 1 minute).

import torch
from sentence_transformers import SentenceTransformer
from sentence_transformers.cross_encoder import CrossEncoder
device = "mps" if torch.backends.mps.is_available() else "cpu"
s1 = SentenceTransformer(str(S1_MODEL), device=device)
ce = CrossEncoder(str(S2_MODEL), max_length=64, device=device)
EMBED_DIM = 384
N_SCAN = SMOKE_POOL if SMOKE else None

def batched_topk_timed(queries, k=RETRIEVE_K, chunk_rows=250_000):
    E = np.load(EMB_NPY, mmap_mode="r"); P = np.load(PID_NPY)
    N = E.shape[0] if N_SCAN is None else min(N_SCAN, E.shape[0])
    t0 = time.perf_counter()
    Q = s1.encode(list(queries), batch_size=EMBED_BATCH, convert_to_numpy=True,
                  normalize_embeddings=True, show_progress_bar=False).astype(np.float32)
    t1 = time.perf_counter()
    nq = len(Q); run_sim = np.full((nq, k), -np.inf, dtype=np.float32); run_idx = np.zeros((nq, k), dtype=np.int64)
    for s in range(0, N, chunk_rows):
        block = np.asarray(E[s:min(s + chunk_rows, N)]).astype(np.float32, copy=False)
        sims = Q @ block.T
        kk = min(k, sims.shape[1])
        part = np.argpartition(-sims, kk - 1, axis=1)[:, :kk]
        cand_sim = np.take_along_axis(sims, part, axis=1); cand_idx = part.astype(np.int64) + s
        all_sim = np.concatenate([run_sim, cand_sim], axis=1); all_idx = np.concatenate([run_idx, cand_idx], axis=1)
        sel = np.argpartition(-all_sim, k - 1, axis=1)[:, :k]
        run_sim = np.take_along_axis(all_sim, sel, axis=1); run_idx = np.take_along_axis(all_idx, sel, axis=1)
    order = np.argsort(-run_sim, axis=1)
    run_sim = np.take_along_axis(run_sim, order, axis=1); run_idx = np.take_along_axis(run_idx, order, axis=1)
    t2 = time.perf_counter()
    return P[run_idx], run_sim, dict(encode_s=t1 - t0, scan_s=t2 - t1)

def rerank_timed(queries, pids, sims):
    """resolve the top-K_REPORT pids to strings, score the pairs, pick top-1 per query."""
    t0 = time.perf_counter()
    top = pids[:, :K_REPORT]; need = {b.decode() for row in top for b in row}
    res = resolve(need); R = dict(zip(res.pid, res.s))
    t1 = time.perf_counter()
    recs = [(i, r + 1, top[i, r].decode(), R.get(top[i, r].decode(), ""), float(sims[i, r]))
            for i in range(len(queries)) for r in range(top.shape[1])]
    df = pd.DataFrame(recs, columns=["uid", "rank", "pid", "cand", "sim"])
    logits = ce.predict([(queries[u], c) for u, c in zip(df.uid, df.cand)],
                        batch_size=PREDICT_BATCH, show_progress_bar=False)
    df["s_model"] = 1.0 / (1.0 + np.exp(-np.asarray(logits, dtype=np.float64)))
    t2 = time.perf_counter()
    top1 = (df.sort_values(["uid", "s_model", "rank"], ascending=[True, False, True], kind="mergesort")
              .groupby("uid").head(1).reset_index(drop=True))
    return df, top1, dict(resolve_s=t1 - t0, score_s=t2 - t1)

pids, sims, t_ret = batched_topk_timed(QUERIES)
cand_df, top1_re, t_rr = rerank_timed(QUERIES, pids, sims)
batch_s = t_ret["encode_s"] + t_ret["scan_s"] + t_rr["resolve_s"] + t_rr["score_s"]
print(f"batch of {len(QUERIES)}: encode {t_ret['encode_s']:.1f}s | scan {t_ret['scan_s']:.1f}s | "
      f"resolve {t_rr['resolve_s']:.1f}s | cross-encoder {t_rr['score_s']:.1f}s | total {batch_s/60:.2f} min")
cand_df.to_csv(OUT / "perquery_neural_remeasured.tsv", sep="\t", index=False)

# Check that this run reproduces the saved top-1 picks.
_cmp = top1_re.merge(neural_top1[["uid_here", "cand", "s_model"]].rename(
    columns={"uid_here": "uid", "cand": "cand_archived", "s_model": "s_archived"}), on="uid")
_same = int((_cmp.cand == _cmp.cand_archived).sum())
print(f"top-1 string identical to the archived run on {_same} of {len(_cmp)} queries; "
      f"max |score difference| {float((_cmp.s_model - _cmp.s_archived).abs().max()):.4f}"
      + ("  (smoke: the scan covers a slice of the index, so disagreement is expected)" if SMOKE else ""))

singles = []
for u, q in zip(SINGLE_UIDS, SINGLE_Q):
    t0 = time.perf_counter()
    p1, s1s, _ = batched_topk_timed([q]); _, t1, _ = rerank_timed([q], p1, s1s)
    singles.append(time.perf_counter() - t0)
    print(f"  uid {u} alone: {singles[-1]:.1f}s -> {t1.cand.iloc[0]!r} score {t1.s_model.iloc[0]:.3f}")

record("neural", float("nan"), batch_s, singles,
       "batch = encode + index scan + resolve + cross-encoder; one alone = a full index pass; "
       "preparation quoted, not re-measured: index embed 84 min, Stage 1 train 111 min, "
       "Stage 2 mine 20.6 min + train 9.7 min")
TIMING[-1].update(encode_s=round(t_ret["encode_s"], 2), scan_s=round(t_ret["scan_s"], 2),
                  resolve_s=round(t_rr["resolve_s"], 2), score_s=round(t_rr["score_s"], 2))
pd.DataFrame(TIMING).to_csv(OUT / "timing.tsv", sep="\t", index=False)
del s1, ce
print("timing.tsv updated with the neural row")

batch of 950: encode 0.5s | scan 155.5s | resolve 0.3s | cross-encoder 4.8s | total 2.69 min
top-1 string identical to the archived run on 950 of 950 queries; max |score difference| 0.0000


  uid 773 alone: 57.7s -> '25 HOSKINS AVENUE WARRAWONG NSW 2502' score 0.917


  uid 114 alone: 60.5s -> '70-74 KENT ROAD ORCHARD HILLS NSW 2748' score 0.997


  uid 125 alone: 60.5s -> '90 GEORGE STREET PARRAMATTA NSW 2150' score 0.930


  uid 456 alone: 60.3s -> '240 MARGARET STREET BRISBANE CITY QLD 4000' score 0.954


  uid 723 alone: 60.3s -> '75-81 DARLINGHURST ROAD POTTS POINT NSW 2011' score 0.998
  bi-encoder + cross-encoder (K=10)    build     nans | batch   161.1s ( 169.6 ms/query) | one alone 59.870s
timing.tsv updated with the neural row


In [ ]:
# Cell 11: accuracy, thresholds and confusion counts, read from the
# per-query files on disk so Part 2 can reuse compute_metrics(). For each
# method: top-1 on the 789 matchable rows (string rule and pid rule),
# recall@1/10/50, AUROC, the best threshold on all 950 rows with its
# confusion counts, and 5-fold cross-validated accuracy (folds balanced by
# population). The same is done for the pipeline's saved scores, plus the
# pipeline at its own tau. A query with no candidate gets score 0.
# The metric functions come from baseline_metrics.py.

sys.path.insert(0, "tools")
from baseline_metrics import confusion, best_threshold, cv_accuracy, auroc, recall_at, wilson

METHOD_ORDER = ["rapidfuzz_ratio", "levenshtein", "jaro_winkler", "tfidf_char3", "double_metaphone"]
FALLBACK_LABEL = {"rapidfuzz_ratio": "RapidFuzz ratio", "levenshtein": "Levenshtein",
                  "jaro_winkler": "Jaro-Winkler", "tfidf_char3": "TF-IDF cosine (character 3-grams)",
                  "double_metaphone": "Double Metaphone + ratio",
                  "neural": "bi-encoder + cross-encoder (K=10)"}

def compute_metrics(folder, neural_df, tau_pipeline, neural_label=None, quiet=False):
    folder = Path(folder)
    methods = [m for m in METHOD_ORDER if (folder / f"perquery_{m}.tsv").exists()]
    assert methods, f"no perquery_<method>.tsv in {folder}"
    labels = dict(FALLBACK_LABEL)
    if (folder / "timing.tsv").exists():
        _t = pd.read_csv(folder / "timing.tsv", sep="\t")
        labels.update(dict(zip(_t.method, _t.label)))
    if neural_label:
        labels["neural"] = neural_label
    allq[["uid", "address", "q", "population", "kind", "truth_pid"]].assign(
        truth_str=[TRUTH_STR.get(int(u), "") for u in allq.uid]).to_csv(folder / "queries.tsv", sep="\t", index=False)

    perq = {}
    for m in methods:
        p = pd.read_csv(folder / f"perquery_{m}.tsv", sep="\t", dtype={"pid": str, "cand": str, "score": np.float32})
        p["pid"] = p.pid.fillna(""); p["cand"] = p.cand.fillna("")
        perq[m] = p
    TOP1 = {}
    for m in methods:
        t1 = perq[m][perq[m]["rank"] == 1][["uid", "pid", "cand", "score"]]
        t = allq[["uid", "address", "population", "kind"]].merge(t1, on="uid", how="left")
        t["pid"] = t.pid.fillna(""); t["cand"] = t.cand.fillna("")
        t["score"] = t.score.astype(np.float64).fillna(0.0)      # no candidate at all -> score 0
        t["is_truth"] = [c == TRUTH_STR.get(int(u), "\x00") for c, u in zip(t.cand, t.uid)]
        TOP1[m] = t
    _n = neural_df.drop(columns=["uid", "is_truth"]).rename(
        columns={"uid_here": "uid", "s_model": "score", "is_truth_here": "is_truth"})
    TOP1["neural"] = _n[["uid", "address", "population", "kind", "pid", "cand", "score", "is_truth"]].sort_values("uid").reset_index(drop=True)

    _all_pids = sorted({p for t in TOP1.values() for p in t.pid if p} | set(TRUTH_PRINCIPAL))
    _pr = resolve(_all_pids); P2P = dict(zip(_pr.pid, _pr.principal))

    METRICS = []
    for name, t in TOP1.items():
        t = t[t.population != "excluded"].copy()
        t.to_csv(folder / f"top1_{name}.tsv", sep="\t", index=False)
        m = (t.population == "matchable").to_numpy(); sc = t.score.to_numpy(dtype=float); ok = t.is_truth.to_numpy()
        truth_pr = np.array([P2P.get(allq.truth_pid[int(u)], allq.truth_pid[int(u)]) for u in t.uid])
        pid_ok = np.array([P2P.get(p, p) if p else "" for p in t.pid]) == truth_pr
        n_m, n_u = int(m.sum()), int((~m).sum())
        row = dict(method=name, label=labels[name], n_matchable=n_m, n_unmatchable=n_u,
                   top1_str=int((ok & m).sum()), top1_pid=int((pid_ok & m).sum()),
                   auroc=auroc(sc[m], sc[~m]))
        row["top1_str_rate"] = row["top1_str"] / n_m; row["top1_pid_rate"] = row["top1_pid"] / n_m
        row["top1_str_ci_lo"], row["top1_str_ci_hi"] = wilson(row["top1_str_rate"], n_m)
        if name in perq:
            for K in (1, 10, 50):
                row[f"recall_{K}"] = recall_at(perq[name], TRUTH_STR, m_uids, K)
            assert row["recall_1"] == row["top1_str"], f"{name}: recall@1 must equal top-1 string"
        row["n_no_candidate"] = int((t.cand == "").sum())
        bt, acc = best_threshold(sc, ok, m); c = confusion(sc, ok, m, bt)
        row.update(best_t=bt, best_accurate=acc, best_accuracy=acc / len(t),
                   **{f"best_{k}": v for k, v in c.items() if k not in ("threshold", "n", "accurate", "accuracy")})
        for kind in ("matchable", "no_number", "no_record"):
            kk = (t.kind == kind).to_numpy()
            row[f"best_right_{kind}"] = int(((sc[kk] >= bt) & ok[kk]).sum()) if kind == "matchable" else int((sc[kk] < bt).sum())
            row[f"n_{kind}"] = int(kk.sum())
        cv = cv_accuracy(sc, ok, m, t.population.to_numpy(), seed=SEED)
        row.update(cv_accurate=cv["accurate"], cv_accuracy=cv["accuracy"],
                   cv_t_min=min(cv["thresholds"]), cv_t_max=max(cv["thresholds"]),
                   cv_fold_accurate=json.dumps(cv["fold_accurate"]), cv_fold_n=json.dumps(cv["fold_n"]))
        if name == "neural" and not np.isnan(tau_pipeline):
            c2 = confusion(sc, ok, m, tau_pipeline)
            row.update(own_tau=tau_pipeline, own_tau_accurate=c2["accurate"], own_tau_accuracy=c2["accuracy"],
                       **{f"own_tau_{k}": v for k, v in c2.items() if k in
                          ("correct_accept", "wrong_accept_wrong_record", "wrong_accept_unmatchable",
                           "rejected_matchable", "correct_reject")})
        METRICS.append(row)

    met = pd.DataFrame(METRICS); met.to_csv(folder / "metrics.tsv", sep="\t", index=False)
    if not quiet:
        pd.set_option("display.width", 200)
        print(f"[{folder}] TOP-1 ON THE MATCHABLE ROWS (string rule) AND ACCEPT/REFUSE ACCURACY OVER ALL ROWS\n")
        show = met[["label", "n_matchable", "top1_str", "top1_str_rate", "top1_pid", "auroc",
                    "best_t", "best_accurate", "best_accuracy", "cv_accurate", "cv_accuracy", "cv_t_min", "cv_t_max"]]
        print(show.round(4).to_string(index=False))
        print("\nCONFUSION COUNTS AT THE BEST THRESHOLD (correct accept / wrong record / unmatchable accepted / matchable refused / correct refuse)")
        for r in METRICS:
            print(f"  {r['label']:44s} {r['best_correct_accept']:4d} / {r['best_wrong_accept_wrong_record']:3d} / "
                  f"{r['best_wrong_accept_unmatchable']:3d} / {r['best_rejected_matchable']:3d} / {r['best_correct_reject']:3d}"
                  f"   right by kind: matchable {r['best_right_matchable']}/{r['n_matchable']}, "
                  f"no_number {r['best_right_no_number']}/{r['n_no_number']}, no_record {r['best_right_no_record']}/{r['n_no_record']}"
                  + (f"   (no candidate: {r['n_no_candidate']})" if r["n_no_candidate"] else ""))
        _nr = met[met.method == "neural"].iloc[0]
        if "own_tau" in met.columns:
            print(f"\nneural pipeline at its own tau {_nr.own_tau:.2f}: {int(_nr.own_tau_accurate)} of {len(TOP1['neural'])} right "
                  f"({_nr.own_tau_accuracy:.4f}); counts {int(_nr.own_tau_correct_accept)} / {int(_nr.own_tau_wrong_accept_wrong_record)} / "
                  f"{int(_nr.own_tau_wrong_accept_unmatchable)} / {int(_nr.own_tau_rejected_matchable)} / {int(_nr.own_tau_correct_reject)}")
        print("\nRECALL@K on the matchable rows (string rule; the neural pipeline's are in 3_train_retriever.ipynb and 4_train_reranker.ipynb)")
        print(met.loc[met.method != "neural", ["label", "recall_1", "recall_10", "recall_50"]]
              .astype({"recall_1": int, "recall_10": int, "recall_50": int}).to_string(index=False))
    return met, TOP1, METRICS, P2P

met, TOP1, METRICS, P2P = compute_metrics(OUT, neural_top1, TAU_PIPELINE)

[output_baseline] TOP-1 ON THE MATCHABLE ROWS (string rule) AND ACCEPT/REFUSE ACCURACY OVER ALL ROWS

                            label  n_matchable  top1_str  top1_str_rate  top1_pid  auroc  best_t  best_accurate  best_accuracy  cv_accurate  cv_accuracy  cv_t_min  cv_t_max
                  RapidFuzz ratio          789       457         0.5792       442 0.5195  0.7333            457         0.4811          455       0.4789    0.6829    0.7333
                      Levenshtein          789       429         0.5437       412 0.5010  0.5610            430         0.4526          425       0.4474    0.5610    0.6140
                     Jaro-Winkler          789       417         0.5285       412 0.5501  0.9003            433         0.4558          417       0.4389    0.8671    0.9043
TF-IDF cosine (character 3-grams)          789       502         0.6362       488 0.5045  0.3927            503         0.5295          501       0.5274    0.3927    0.4957
         Double Metaphone + ratio

In [ ]:
# Cell 12: the followed address through every method: each method's top 3
# with scores, whether the top-1 is the true record, and whether it is
# accepted at that method's best threshold. Reads the per-query files, so
# Part 2 can reuse trace().

def trace(folder, met, TOP1, uid=FOLLOW_UID):
    folder = Path(folder)
    print(f"[{folder}]\nquery : {allq.q[uid]!r}\ntruth : {TRUTH_STR[uid]!r}\n")
    for name in met.method:
        if name == "neural":
            continue
        pq_ = pd.read_csv(folder / f"perquery_{name}.tsv", sep="\t", dtype={"pid": str, "cand": str})
        g = pq_[(pq_.uid == uid) & (pq_["rank"] <= 3)].sort_values("rank")
        bt = float(met.loc[met.method == name, "best_t"].iloc[0])
        lab = met.loc[met.method == name, "label"].iloc[0]
        print(f"{lab} (best threshold {bt:.3f})")
        if g.empty:
            print("   no candidate (empty block)\n"); continue
        for r in g.itertuples():
            mark = "TRUE" if r.cand == TRUTH_STR[uid] else "    "
            print(f"   {r.rank}. {r.score:.3f}  {mark}  {r.cand}")
        top = TOP1[name]; s0 = float(top.loc[top.uid == uid, "score"].iloc[0])
        print(f"   top-1 {'accepted' if s0 >= bt else 'refused'} at the best threshold\n")
    nr = TOP1["neural"]; nrow = nr[nr.uid == uid].iloc[0]
    bt = float(met.loc[met.method == "neural", "best_t"].iloc[0])
    lab = met.loc[met.method == "neural", "label"].iloc[0]
    print(f"{lab} (archived run; best threshold {bt:.3f}, own tau {TAU_PIPELINE:.2f})")
    print(f"   1. {nrow.score:.3f}  {'TRUE' if nrow.is_truth else '    '}  {nrow.cand}")
    print(f"   {'accepted' if nrow.score >= bt else 'refused'} at the best threshold; "
          f"{'accepted' if nrow.score >= TAU_PIPELINE else 'refused'} at the pipeline's own tau")

trace(OUT, met, TOP1)

[output_baseline]
query : '25 HOSKINS AVENUE KEMBLAWARRA NSW 2505'
truth : '25 HOSKINS AVENUE WARRAWONG NSW 2502'

RapidFuzz ratio (best threshold 0.733)
   1. 0.838  TRUE  25 HOSKINS AVENUE WARRAWONG NSW 2502
   2. 0.822        5 HOSKINS AVENUE WARRAWONG NSW 2502
   3. 0.822        2 HOSKINS AVENUE WARRAWONG NSW 2502
   top-1 accepted at the best threshold

Levenshtein (best threshold 0.561)
   1. 0.737  TRUE  25 HOSKINS AVENUE WARRAWONG NSW 2502
   2. 0.711        35 HOSKINS AVENUE WARRAWONG NSW 2502
   3. 0.711        28 HOSKINS AVENUE WARRAWONG NSW 2502
   top-1 accepted at the best threshold

Jaro-Winkler (best threshold 0.900)
   1. 0.916  TRUE  25 HOSKINS AVENUE WARRAWONG NSW 2502
   2. 0.910        25 HOSKIN AVENUE KIDMAN PARK SA 5025
   3. 0.901        25 HOSKINS AVENUE LITHGOW NSW 2790
   top-1 accepted at the best threshold

TF-IDF cosine (character 3-grams) (best threshold 0.393)
   1. 0.690        25 LAWARRA STREET PORT KEMBLA NSW 2505
   2. 0.651  TRUE  25 HOSKINS AVENUE 

Double Metaphone + ratio (best threshold 0.724)
   1. 0.830        2 SECOND AVENUE PORT KEMBLA NSW 2505
   2. 0.830        5 SECOND AVENUE PORT KEMBLA NSW 2505
   3. 0.830        2 SIXTH AVENUE PORT KEMBLA NSW 2505
   top-1 accepted at the best threshold

bi-encoder + cross-encoder (K=10) (archived run; best threshold 0.299, own tau 0.83)
   1. 0.917  TRUE  25 HOSKINS AVENUE WARRAWONG NSW 2502
   accepted at the best threshold; accepted at the pipeline's own tau


In [ ]:
# Cell 13: write a manifest of what this run read and produced, then close
# the database.

import subprocess, platform, rapidfuzz, sklearn, metaphone
try:
    _git = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    _git = ""
manifest = dict(
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), smoke=SMOKE, git_hash=_git,
    test_file=str(MISS), test_sha1=TEST_SHA1,
    n_rows=int(len(allq)), n_matchable=int(len(m_uids)), n_unmatchable=int(len(u_uids)),
    n_no_number=int((allq.kind == "no_number").sum()), n_no_record=int((allq.kind == "no_record").sum()),
    n_excluded=int((allq.population == "excluded").sum()),
    pool_rows=int(N_POOL), pool_equals_index=bool(not SMOKE), pid_file_sha1=PID_SHA1, render_sha1=RENDER_SHA1,
    prep_seconds=PREP, k_keep=K_KEEP, chunk_rows=CHUNK_ROWS, seed=SEED, n_single=N_SINGLE,
    single_uids=[int(x) for x in SINGLE_UIDS], tfidf=dict(features=TFIDF_FEATURES, ngram=list(TFIDF_NGRAM),
    analyzer="char_wb"), neural_predictions=str(S2_PRED), tau_pipeline=TAU_PIPELINE,
    methods=list(RESULTS.keys()) + ["neural"], labels=METHOD_LABEL,
    versions=dict(python=platform.python_version(), rapidfuzz=rapidfuzz.__version__,
                  sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__),
    machine=platform.machine(),
    best_thresholds={r["method"]: r["best_t"] for r in METRICS},
    best_accurate={r["method"]: int(r["best_accurate"]) for r in METRICS},
    cv_accurate={r["method"]: int(r["cv_accurate"]) for r in METRICS},
)
(OUT / "baseline_manifest.json").write_text(json.dumps(manifest, indent=2))
con.close()
print(json.dumps({k: manifest[k] for k in ("timestamp", "test_sha1", "pool_rows", "best_accurate", "cv_accurate")}, indent=1))
print(f"\nwrote {OUT/'baseline_manifest.json'}; database closed")

{
 "timestamp": "2026-09-09 19:37:38",
 "test_sha1": "d48fdfe227fd6ef7828384f020e3042d18873101",
 "pool_rows": 15901232,
 "best_accurate": {
  "rapidfuzz_ratio": 457,
  "levenshtein": 430,
  "jaro_winkler": 433,
  "tfidf_char3": 503,
  "double_metaphone": 433,
  "neural": 745
 },
 "cv_accurate": {
  "rapidfuzz_ratio": 455,
  "levenshtein": 425,
  "jaro_winkler": 417,
  "tfidf_char3": 501,
  "double_metaphone": 428,
  "neural": 744
 }
}

wrote output_baseline/baseline_manifest.json; database closed


## Part 2: the same five methods, searching only records with the query's street number

Each method now looks only at the G-NAF records that carry the query's
street number. The rule:

- The query's street-number token is read as a range. A single number is a range of one.
- Both parts of a slash token count, so "6/1" blocks on 6 and on 1.
- Long dashes and "317 - 321" are read as ranges.
- Letter suffixes are ignored, so "6A" is 6.
- A record is in the block when its own number range overlaps the query's.
- A query whose block is empty gets no candidate. Its score is 0 and it is deemed unmatchable.

The neural pipeline is not run again and not blocked. Its rows are the saved
ones, labelled "no block". Part 2 needs the setup Cells 1 to 8, Cells 11 and
12 (they define the metric and trace functions) and Part 1's output folder.
It writes to `output_baseline_blocked/` and leaves Part 1's folder as it is.

In [ ]:
# Cell 14: Part 2 configuration, the database reopened read-only (Cell 13
# closed it), and the two street-number parsers.
# query_intervals(q) returns the query's street number(s) as (first, last)
# ranges. The number that counts is the one right before the first street
# name word. Unit and lot words are skipped, and a number with no word after
# it (the postcode) does not count, so a query with no street number gives [].
# pool_intervals() parses pool strings with one regular expression. RENDER
# built each string from the key and number_last, so this parse is exact.

OUT_B = Path(_env("B_OUT_BLOCKED", "scratch/baseline_smoke_blocked" if SMOKE else "output_baseline_blocked"))
_o = OUT_B.resolve()
for _p in PROTECTED:
    _r = Path(_p).resolve()
    assert _o != _r and _r not in _o.parents, f"OUT_B={OUT_B} is inside protected {_p}; refusing"
assert _o != OUT.resolve(), "the blocked run must not write into Part 1's folder"
OUT_B.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(DB_PATH), read_only=True)

UNIT_WORDS = {"UNIT", "U", "LOT", "LEVEL", "L", "LVL", "SHOP", "SUITE", "STE", "FLAT", "APT",
              "APARTMENT", "VILLA", "FLOOR", "FL", "ROOM", "OFFICE", "KIOSK", "FACTORY",
              "WAREHOUSE", "BLOCK", "BUILDING", "TOWER", "TENANCY", "PO", "BOX", "GPO", "RMB", "RSD"}
_DASHES = re.compile("[‐‑‒–—−]")
_NUM = re.compile(r"(\d+)[A-Z]?(?:-(\d+)[A-Z]?)?")

def query_intervals(q):
    s = _DASHES.sub("-", q)
    s = re.sub(r"(\d+[A-Z]?)\s*-\s*(\d+[A-Z]?)", r"\1-\2", s)
    last = None
    for tok in s.split():
        iv = []
        for part in tok.split("/"):
            m = _NUM.fullmatch(part)
            if m:
                a = int(m.group(1)); b = int(m.group(2)) if m.group(2) else a
                iv.append((min(a, b), max(a, b)))
        if iv:
            last = iv; continue
        if tok in UNIT_WORDS:
            continue
        if last is not None and re.search(r"[A-Z]", tok):
            return last
    return []

def pool_intervals(strings):
    ex = pd.Series(strings).str.extract(r"^(?:LOT )?(\d+)[A-Z]?(?:-(\d+)[A-Z]?)?\s")
    first = pd.to_numeric(ex[0], errors="coerce").to_numpy()
    last = pd.to_numeric(ex[1], errors="coerce").to_numpy()
    last = np.where(np.isnan(last), first, last)
    lo = np.where(np.isnan(first), -1, np.minimum(first, last)).astype(np.int64)
    hi = np.where(np.isnan(first), -1, np.maximum(first, last)).astype(np.int64)
    return lo, hi

for _s in [norm(FOLLOW), "29/187 THOMAS STREET SYDNEY NSW 2000", "UNIT 10 92-100 BELMORE RD NORTH RIVERWOOD NSW 2210",
           "6/1 MAXIM STREET WEST RYDE NSW 2144", "317 - 321 BARRENJOEY ROAD NEWPORT BEACH NSW 2106",
           "97–99 GOULBURN STREET SYDNEY NSW 2000", "LOT 102 WONGA ROAD PRESTONS NSW 2170",
           "6A BERGHOFER COURT CHARNWOOD ACT 2615", "CONSETT AVENUE BONDI BEACH NSW 2026"]:
    print(f"  {_s!r:58s} -> {query_intervals(_s)}")
print(f"  pool: {POOL[0]!r} -> {tuple(int(x[0]) for x in pool_intervals([POOL[0]]))}")

  '25 HOSKINS AVENUE KEMBLAWARRA NSW 2505'                   -> [(25, 25)]
  '29/187 THOMAS STREET SYDNEY NSW 2000'                     -> [(29, 29), (187, 187)]
  'UNIT 10 92-100 BELMORE RD NORTH RIVERWOOD NSW 2210'       -> [(92, 100)]
  '6/1 MAXIM STREET WEST RYDE NSW 2144'                      -> [(6, 6), (1, 1)]
  '317 - 321 BARRENJOEY ROAD NEWPORT BEACH NSW 2106'         -> [(317, 321)]
  '97–99 GOULBURN STREET SYDNEY NSW 2000'                    -> [(97, 99)]
  'LOT 102 WONGA ROAD PRESTONS NSW 2170'                     -> [(102, 102)]
  '6A BERGHOFER COURT CHARNWOOD ACT 2615'                    -> [(6, 6)]
  'CONSETT AVENUE BONDI BEACH NSW 2026'                      -> []
  pool: '6 PACKHAM PLACE CHARNWOOD ACT 2615' -> (6, 6)


In [ ]:
# Cell 15: the block index. Parse every pool string's number range once and
# sort the pool by first number, so a block of single-number records is one
# contiguous slice. Range records (first < last) go in a small side table.
# block_rows() returns the original pool rows whose range overlaps any of
# the query's intervals. Then the ceiling: how many matchable queries have
# their true record inside their block, which is the most any blocked method
# can get right. Writes OUT_B/queries_numbers.tsv.

PREP_B = {}
t0 = time.perf_counter()
POOL_LO, POOL_HI = pool_intervals(POOL)
PREP_B["parse_s"] = time.perf_counter() - t0
t0 = time.perf_counter()
ORDER = np.argsort(POOL_LO, kind="stable")
SORTED_LO = POOL_LO[ORDER]
POOL_SORTED = [POOL[i] for i in ORDER]
RANGE_ROWS = np.flatnonzero((POOL_HI > POOL_LO) & (POOL_LO >= 0))
RANGE_LO, RANGE_HI = POOL_LO[RANGE_ROWS], POOL_HI[RANGE_ROWS]
PREP_B["index_s"] = time.perf_counter() - t0
print(f"pool parsed in {PREP_B['parse_s']:.0f}s, sorted in {PREP_B['index_s']:.0f}s: "
      f"{int((POOL_LO < 0).sum()):,} rows without a number, {len(RANGE_ROWS):,} range records, "
      f"{len(np.unique(SORTED_LO[SORTED_LO >= 0])):,} distinct first numbers")

def block_rows(intervals):
    """original pool rows whose number range overlaps any query interval (sorted, unique)"""
    parts = []
    for qf, ql in intervals:
        lo = np.searchsorted(SORTED_LO, qf, side="left"); hi = np.searchsorted(SORTED_LO, ql, side="right")
        parts.append(ORDER[lo:hi])
        parts.append(RANGE_ROWS[(RANGE_LO < qf) & (RANGE_HI >= qf)])
    if not parts:
        return np.zeros(0, dtype=np.int64)
    return np.unique(np.concatenate(parts))

QUERIES = allq.q.tolist()
Q_IV = [query_intervals(q) for q in QUERIES]
_truth_row = {int(u): int(r) for u, r in zip(m_uids, row_of_pid(allq.loc[m_uids, "truth_pid"].to_numpy().astype("S32")))}
_rows = []
for u, iv in zip(allq.uid, Q_IV):
    b = block_rows(iv)
    inb = (int(np.searchsorted(b, _truth_row[u]) < len(b)) and bool(b[np.searchsorted(b, _truth_row[u])] == _truth_row[u])) if u in _truth_row else None
    _rows.append(dict(uid=int(u), q=QUERIES[u], population=allq.population[u], kind=allq.kind[u],
                      intervals=json.dumps(iv), n_intervals=len(iv), block_size=int(len(b)),
                      truth_in_block=inb, truth_str=TRUTH_STR.get(int(u), "")))
qnum = pd.DataFrame(_rows); qnum.to_csv(OUT_B / "queries_numbers.tsv", sep="\t", index=False)
_m = qnum[qnum.population == "matchable"]
CEILING = dict(matchable=int(len(_m)), truth_in_block=int(_m.truth_in_block.sum()),
               no_number=int((qnum.n_intervals == 0).sum()), empty_block=int((qnum.block_size == 0).sum()),
               block_min=int(qnum.block_size.min()), block_median=int(qnum.block_size.median()),
               block_max=int(qnum.block_size.max()), block_total=int(qnum.block_size.sum()))
print(f"queries with no street number: {CEILING['no_number']}; with an empty block: {CEILING['empty_block']}")
print(f"block sizes: min {CEILING['block_min']:,}  median {CEILING['block_median']:,}  max {CEILING['block_max']:,}  "
      f"total over {len(qnum)} queries {CEILING['block_total']:,} (was {len(qnum) * N_POOL:,} without the block)")
print(f"CEILING: {CEILING['truth_in_block']} of {CEILING['matchable']} matchable queries have their true record inside the block")
for r in _m[~_m.truth_in_block.astype(bool)].itertuples():
    print(f"   outside: {r.q!r} -> truth {r.truth_str!r}  intervals {r.intervals}")
print(f"followed address: intervals {Q_IV[FOLLOW_UID]}, block of {int(qnum.block_size[FOLLOW_UID]):,} records, truth inside: {qnum.truth_in_block[FOLLOW_UID]}")

pool parsed in 29s, sorted in 7s: 2,888 rows without a number, 1,138,980 range records, 23,255 distinct first numbers


queries with no street number: 0; with an empty block: 0
block sizes: min 1  median 133,397  max 8,500,127  total over 950 queries 272,661,288 (was 15,106,170,400 without the block)
CEILING: 788 of 789 matchable queries have their true record inside the block
   outside: '8/4656 KIPPAX ST SURRY HILLS NSW 2010' -> truth '46-56 KIPPAX STREET SURRY HILLS NSW 2010'  intervals [[8, 8], [4656, 4656]]
followed address: intervals [(25, 25)], block of 247,834 records, truth inside: True


In [ ]:
# Cell 16: the blocked search engines. block_strings() collects a query's
# block from the sorted pool and the range side table.
# RapidFuzz: one thread per query, N_THREADS queries at a time.
# TF-IDF: the full pool matrix is built once and held in memory (this is the
# preparation), then a block is a row slice of it times the query vector.
# Double Metaphone: the coded pool, sorted the same way, with the ratio.
# Each search returns the top-K rows and scores, ordered by score then row.
# An empty block returns nothing.

import scipy.sparse as sp

def block_strings(intervals, sorted_list, base_list):
    rows, strs = [], []
    for qf, ql in intervals:
        lo = np.searchsorted(SORTED_LO, qf, side="left"); hi = np.searchsorted(SORTED_LO, ql, side="right")
        rows.append(ORDER[lo:hi]); strs.extend(sorted_list[lo:hi])
        extra = RANGE_ROWS[(RANGE_LO < qf) & (RANGE_HI >= qf)]
        rows.append(extra); strs.extend(base_list[i] for i in extra)
    if not rows:
        return np.zeros(0, dtype=np.int64), []
    rows = np.concatenate(rows)
    if len(intervals) > 1:
        _, keep = np.unique(rows, return_index=True); keep.sort()
        rows = rows[keep]; strs = [strs[i] for i in keep]
    return rows, strs

def topk_of(rows, sims, k=K_KEEP):
    if len(rows) == 0:
        return rows, np.zeros(0, dtype=np.float32)
    kk = min(k, len(sims)); part = np.argpartition(-sims, kk - 1)[:kk]
    r, s = rows[part], sims[part]
    o = np.lexsort((r, -s))
    return r[o], s[o].astype(np.float32)

def ratio_scorer(scorer, scale):
    def f(i, q, rows, strs):
        return process.cdist([q], strs, scorer=scorer, workers=1, dtype=np.float32)[0] * scale
    return f

def blocked_batch(scorefn, queries, ivs, sorted_list, base_list, threads=N_THREADS):
    def one(i):
        rows, strs = block_strings(ivs[i], sorted_list, base_list)
        return topk_of(rows, scorefn(i, queries[i], rows, strs)) if len(rows) else (rows, np.zeros(0, np.float32))
    t0 = time.perf_counter()
    with ThreadPoolExecutor(threads) as ex:
        out = list(ex.map(one, range(len(queries))))
    return out, time.perf_counter() - t0

def blocked_single(scorefn, i, q, iv, sorted_list, base_list):
    t0 = time.perf_counter()
    rows, strs = block_strings(iv, sorted_list, base_list)
    res = topk_of(rows, scorefn(i, q, rows, strs)) if len(rows) else (rows, np.zeros(0, np.float32))
    return res, time.perf_counter() - t0

def build_tfidf_matrix(pool, chunk_rows=CHUNK_ROWS):
    """idf pass (also counts nonzeros per block), then a fill pass into a preallocated CSR."""
    t0 = time.perf_counter()
    df = np.zeros(TFIDF_FEATURES, dtype=np.int64); nnz = []
    for s in range(0, len(pool), chunk_rows):
        X = HV.transform(pool[s:s + chunk_rows])
        df += np.bincount(X.indices, minlength=TFIDF_FEATURES); nnz.append(X.nnz)
    idf = (np.log((1 + len(pool)) / (1 + df)) + 1).astype(np.float32)
    total = int(sum(nnz)); assert total < 2**31 - 1
    data = np.empty(total, dtype=np.float32); indices = np.empty(total, dtype=np.int32)
    indptr = np.zeros(len(pool) + 1, dtype=np.int32)
    pos, row = 0, 0
    for s in range(0, len(pool), chunk_rows):
        X = _vec(pool[s:s + chunk_rows], idf); m = X.nnz
        data[pos:pos + m] = X.data; indices[pos:pos + m] = X.indices
        indptr[row + 1:row + 1 + X.shape[0]] = X.indptr[1:] + pos
        pos += m; row += X.shape[0]
    assert pos == total and row == len(pool)
    X = sp.csr_matrix((data, indices, indptr), shape=(len(pool), TFIDF_FEATURES))
    return X, idf, time.perf_counter() - t0

def tfidf_scorer(X, Q):
    QT = Q.T.tocsc()
    def f(i, q, rows, strs):
        return np.asarray((X[rows] @ QT[:, i]).todense(), dtype=np.float32).ravel()
    return f

print("blocked engines ready")

blocked engines ready


In [ ]:
# Cell 17: run the five blocked methods, as Cell 9 does, but each query sees
# only its block. Writes OUT_B/perquery_<method>.tsv (a query with an empty
# block has no rows) and OUT_B/timing.tsv. Preparation is the shared parse
# and sort (Cell 15) plus each method's own. The neural timing row is copied
# from Part 1, labelled "no block". Building the TF-IDF matrix takes about
# 6 minutes. Each search takes from seconds to about 2 minutes.

LABEL_B = {"rapidfuzz_ratio": "RapidFuzz ratio, street-number block",
           "levenshtein": "Levenshtein, street-number block",
           "jaro_winkler": "Jaro-Winkler, street-number block",
           "tfidf_char3": "TF-IDF cosine (character 3-grams), street-number block",
           "double_metaphone": "Double Metaphone + ratio, street-number block",
           "neural": "bi-encoder + cross-encoder (K=10), no block"}
_rng = np.random.default_rng(SEED)
_others = [u for u in m_uids if u != FOLLOW_UID]
SINGLE_UIDS = [FOLLOW_UID] + sorted(_rng.choice(_others, size=N_SINGLE - 1, replace=False).tolist())
if (OUT / "baseline_manifest.json").exists():
    assert SINGLE_UIDS == json.loads((OUT / "baseline_manifest.json").read_text())["single_uids"], "single uids differ from Part 1's"
print("one-query-alone timing on uids", SINGLE_UIDS)
SHARED_PREP = PREP_B["parse_s"] + PREP_B["index_s"]

RESULTS_B, TIMING_B = {}, []
def write_perquery_b(name, out):
    recs = []
    for u, (rows, scores) in enumerate(out):
        for r, (i, s) in enumerate(zip(rows, scores)):
            recs.append((u, r + 1, POOL_PIDS[i].decode(), POOL[i], float(s)))
    df = pd.DataFrame(recs, columns=["uid", "rank", "pid", "cand", "score"])
    df.to_csv(OUT_B / f"perquery_{name}.tsv", sep="\t", index=False)
    return df

def record_b(name, build_s, batch_s, singles, note=""):
    TIMING_B.append(dict(method=name, label=LABEL_B[name], build_s=round(build_s, 3),
                         batch_s=round(batch_s, 3), per_query_ms=round(batch_s / len(QUERIES) * 1000, 3),
                         single_s_mean=round(float(np.mean(singles)), 5),
                         single_s_runs=json.dumps([round(x, 5) for x in singles]), note=note))
    print(f"  {LABEL_B[name]:56s} prep {build_s:6.1f}s | batch {batch_s:6.2f}s ({batch_s/len(QUERIES)*1000:6.2f} ms/query) | one alone {np.mean(singles)*1000:.1f} ms")

def check_single(name, out, u, res):
    rows, scores = out[u]; r1, s1 = res
    assert (len(rows) == 0 and len(r1) == 0) or (r1[0] == rows[0] or abs(float(s1[0]) - float(scores[0])) < 1e-6), \
        f"{name}: one-query search disagrees with the batch on uid {u}"

for name, scorer, scale in [("rapidfuzz_ratio", fuzz.ratio, 0.01),
                            ("levenshtein", Levenshtein.normalized_similarity, 1.0),
                            ("jaro_winkler", JaroWinkler.similarity, 1.0)]:
    f = ratio_scorer(scorer, scale)
    out, secs = blocked_batch(f, QUERIES, Q_IV, POOL_SORTED, POOL)
    RESULTS_B[name] = out; write_perquery_b(name, out)
    singles = []
    for u in SINGLE_UIDS:
        res, dt = blocked_single(f, u, QUERIES[u], Q_IV[u], POOL_SORTED, POOL); singles.append(dt); check_single(name, out, u, res)
    record_b(name, SHARED_PREP, secs, singles, "prep = parse the pool's numbers + sort")

X_TFIDF, idf_b, mat_s = build_tfidf_matrix(POOL)
Q_B = _vec(QUERIES, idf_b)
f = tfidf_scorer(X_TFIDF, Q_B)
out, secs = blocked_batch(f, QUERIES, Q_IV, POOL_SORTED, POOL, threads=1)
RESULTS_B["tfidf_char3"] = out; write_perquery_b("tfidf_char3", out)
singles = []
for u in SINGLE_UIDS:
    res, dt = blocked_single(f, u, QUERIES[u], Q_IV[u], POOL_SORTED, POOL); singles.append(dt); check_single("tfidf_char3", out, u, res)
record_b("tfidf_char3", SHARED_PREP + mat_s, secs, singles,
         f"prep = parse + sort + the full TF-IDF matrix held in memory ({X_TFIDF.nnz:,} nonzeros); batch single-threaded")
del X_TFIDF, f

POOL_DM, dm_s, n_vocab = dm_encode_pool(POOL)
t0 = time.perf_counter(); POOL_DM_SORTED = [POOL_DM[i] for i in ORDER]; dm_sort_s = time.perf_counter() - t0
Q_DM = dm_encode(QUERIES)
f = ratio_scorer(fuzz.ratio, 0.01)
out, secs = blocked_batch(f, Q_DM, Q_IV, POOL_DM_SORTED, POOL_DM)
RESULTS_B["double_metaphone"] = out; write_perquery_b("double_metaphone", out)
singles = []
for u in SINGLE_UIDS:
    res, dt = blocked_single(f, u, Q_DM[u], Q_IV[u], POOL_DM_SORTED, POOL_DM); singles.append(dt); check_single("double_metaphone", out, u, res)
record_b("double_metaphone", SHARED_PREP + dm_s + dm_sort_s, secs, singles,
         f"prep = parse + sort + coding the pool ({n_vocab:,} distinct words) + reorder")
del POOL_DM, POOL_DM_SORTED, f

_t1 = pd.read_csv(OUT / "timing.tsv", sep="\t")
_nr = _t1[_t1.method == "neural"]
if len(_nr):
    _nr = _nr.iloc[0].to_dict(); _nr["label"] = LABEL_B["neural"]
    _nr["note"] = "copied from Part 1 (the pipeline is not run again and not blocked); " + str(_nr.get("note", ""))
    TIMING_B.append(_nr)
pd.DataFrame(TIMING_B).to_csv(OUT_B / "timing.tsv", sep="\t", index=False)
print(f"\nwrote {len(RESULTS_B)} per-query files and timing.tsv to {OUT_B}")

one-query-alone timing on uids [773, 114, 125, 456, 723]


  RapidFuzz ratio, street-number block                     prep   36.0s | batch  38.37s ( 40.39 ms/query) | one alone 51.3 ms


  Levenshtein, street-number block                         prep   36.0s | batch  40.38s ( 42.50 ms/query) | one alone 76.9 ms


  Jaro-Winkler, street-number block                        prep   36.0s | batch  38.06s ( 40.07 ms/query) | one alone 66.1 ms


  TF-IDF cosine (character 3-grams), street-number block   prep  393.8s | batch 133.54s (140.57 ms/query) | one alone 62.0 ms


  Double Metaphone + ratio, street-number block            prep   60.0s | batch  37.81s ( 39.80 ms/query) | one alone 39.9 ms

wrote 5 per-query files and timing.tsv to output_baseline_blocked


In [ ]:
# Cell 18: Cell 11's metrics and Cell 12's trace on the blocked folder, a
# check that every empty-block query is deemed unmatchable at the best
# threshold, and
# a no-block against block table read from the two metrics.tsv files.

met_b, TOP1_B, METRICS_B, P2P_B = compute_metrics(OUT_B, neural_top1, TAU_PIPELINE, neural_label=LABEL_B["neural"])
for _m in met_b.method:
    if _m == "neural":
        continue
    _t = TOP1_B[_m]; _bt = float(met_b.loc[met_b.method == _m, "best_t"].iloc[0])
    _empty = _t[_t.cand == ""]
    assert (_empty.score < _bt).all(), f"{_m}: an empty-block query is accepted at the best threshold"
print(f"\nempty-block queries ({int((TOP1_B['rapidfuzz_ratio'].cand == '').sum())}) are refused at every method's best threshold: checked")
print()
trace(OUT_B, met_b, TOP1_B)

_a = pd.read_csv(OUT / "metrics.tsv", sep="\t").set_index("method")
_b = met_b.set_index("method")
print("\nNO BLOCK AGAINST BLOCK (counts; of 789 for top-1 and recall, of 950 for accuracy)")
print(f"  {'method':18s} {'top-1':>13s} {'recall@10':>13s} {'best':>13s} {'cross-val':>13s} {'AUROC':>15s}")
for _m in METHOD_ORDER:
    if _m in _a.index and _m in _b.index:
        print(f"  {_m:18s} {int(_a.top1_str[_m]):5d} -> {int(_b.top1_str[_m]):4d}  "
              f"{int(_a.recall_10[_m]):5d} -> {int(_b.recall_10[_m]):4d}  "
              f"{int(_a.best_accurate[_m]):5d} -> {int(_b.best_accurate[_m]):4d}  "
              f"{int(_a.cv_accurate[_m]):5d} -> {int(_b.cv_accurate[_m]):4d}  "
              f"{_a.auroc[_m]:.3f} -> {_b.auroc[_m]:.3f}")
print(f"  {'neural (no block)':18s} {int(_a.top1_str['neural']):5d}          {'':13s} {int(_a.best_accurate['neural']):5d}          {int(_a.cv_accurate['neural']):5d}          {_a.auroc['neural']:.3f}")

[output_baseline_blocked] TOP-1 ON THE MATCHABLE ROWS (string rule) AND ACCEPT/REFUSE ACCURACY OVER ALL ROWS

                                                 label  n_matchable  top1_str  top1_str_rate  top1_pid  auroc  best_t  best_accurate  best_accuracy  cv_accurate  cv_accuracy  cv_t_min  cv_t_max
                  RapidFuzz ratio, street-number block          789       599         0.7592       570 0.7778  0.7674            637         0.6705          631       0.6642    0.7568    0.7816
                      Levenshtein, street-number block          789       583         0.7389       558 0.7427  0.6140            610         0.6421          606       0.6379    0.6140    0.6250
                     Jaro-Winkler, street-number block          789       567         0.7186       541 0.7469  0.8992            585         0.6158          582       0.6126    0.8977    0.8993
TF-IDF cosine (character 3-grams), street-number block          789       595         0.7541       568 0.7676  0.4

Double Metaphone + ratio, street-number block (best threshold 0.780)
   1. 0.830        25 HOSKIN AVENUE KIDMAN PARK SA 5025
   2. 0.824  TRUE  25 HOSKINS AVENUE WARRAWONG NSW 2502
   3. 0.816        25 ZIONS AVENUE MALABAR NSW 2036
   top-1 accepted at the best threshold

bi-encoder + cross-encoder (K=10), no block (archived run; best threshold 0.299, own tau 0.83)
   1. 0.917  TRUE  25 HOSKINS AVENUE WARRAWONG NSW 2502
   accepted at the best threshold; accepted at the pipeline's own tau

NO BLOCK AGAINST BLOCK (counts; of 789 for top-1 and recall, of 950 for accuracy)
  method                     top-1     recall@10          best     cross-val           AUROC
  rapidfuzz_ratio      457 ->  599    522 ->  700    457 ->  637    455 ->  631  0.520 -> 0.778
  levenshtein          429 ->  583    508 ->  694    430 ->  610    425 ->  606  0.501 -> 0.743
  jaro_winkler         417 ->  567    528 ->  686    433 ->  585    417 ->  582  0.550 -> 0.747
  tfidf_char3          502 ->  595    537

In [ ]:
# Cell 19: write a manifest of what Part 2 read and produced, with the block
# rule and its ceiling, then close the database.

import subprocess, platform, rapidfuzz, sklearn
try:
    _git = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    _git = ""
manifest_b = dict(
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), smoke=SMOKE, git_hash=_git, variant="street_number_block",
    block_rule=("the query's street-number token as a range (a single number is a range of one); both parts of a "
                "slash token; long dashes and 'a - b' read as ranges; letter suffixes ignored; a record is in the "
                "block when its number range overlaps the query's; an empty block gives no candidate (score 0, refused)"),
    ceiling=CEILING, part1_folder=str(OUT),
    test_file=str(MISS), test_sha1=TEST_SHA1,
    n_rows=int(len(allq)), n_matchable=int(len(m_uids)), n_unmatchable=int(len(u_uids)),
    n_no_number=int((allq.kind == "no_number").sum()), n_no_record=int((allq.kind == "no_record").sum()),
    n_excluded=int((allq.population == "excluded").sum()),
    pool_rows=int(N_POOL), pool_equals_index=bool(not SMOKE), pid_file_sha1=PID_SHA1, render_sha1=RENDER_SHA1,
    prep_seconds=PREP_B, k_keep=K_KEEP, seed=SEED, n_single=N_SINGLE, n_threads=N_THREADS,
    single_uids=[int(x) for x in SINGLE_UIDS],
    tfidf=dict(features=TFIDF_FEATURES, ngram=list(TFIDF_NGRAM), analyzer="char_wb"),
    neural_predictions=str(S2_PRED), tau_pipeline=TAU_PIPELINE,
    methods=list(RESULTS_B.keys()) + ["neural"], labels=LABEL_B,
    versions=dict(python=platform.python_version(), rapidfuzz=rapidfuzz.__version__,
                  sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__),
    machine=platform.machine(),
    best_thresholds={r["method"]: r["best_t"] for r in METRICS_B},
    best_accurate={r["method"]: int(r["best_accurate"]) for r in METRICS_B},
    cv_accurate={r["method"]: int(r["cv_accurate"]) for r in METRICS_B},
)
(OUT_B / "baseline_manifest.json").write_text(json.dumps(manifest_b, indent=2))
con.close()
print(json.dumps({k: manifest_b[k] for k in ("timestamp", "variant", "ceiling", "best_accurate", "cv_accurate")}, indent=1))
print(f"\nwrote {OUT_B/'baseline_manifest.json'}; database closed")

{
 "timestamp": "2026-09-09 20:48:25",
 "variant": "street_number_block",
 "ceiling": {
  "matchable": 789,
  "truth_in_block": 788,
  "no_number": 0,
  "empty_block": 0,
  "block_min": 1,
  "block_median": 133397,
  "block_max": 8500127,
  "block_total": 272661288
 },
 "best_accurate": {
  "rapidfuzz_ratio": 637,
  "levenshtein": 610,
  "jaro_winkler": 585,
  "tfidf_char3": 632,
  "double_metaphone": 572,
  "neural": 745
 },
 "cv_accurate": {
  "rapidfuzz_ratio": 631,
  "levenshtein": 606,
  "jaro_winkler": 582,
  "tfidf_char3": 630,
  "double_metaphone": 559,
  "neural": 744
 }
}

wrote output_baseline_blocked/baseline_manifest.json; database closed
